# 15.4 Linked Lists

**Prerequisites:** 15.1 Complexity Analysis, 15.2 Python's Built-ins, 05 OOPs  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Nodes and references - a structure with **no contiguous memory**
- Singly linked: build, traverse, insert, delete
- 🔴 The honest truth: why you will rarely use one in Python, and why interviewers ask anyway
- **Reversal** - iteratively and recursively
- **Floyd's cycle detection** - the fast/slow pointer from 15.3
- The **dummy head** trick that removes half the edge cases
- Doubly linked lists, and the LRU cache they enable
- Interview questions, worked

---

## The idea

An array stores elements **next to each other**, so position is computable. A linked list stores each element in its own node that **points at the next one**.

```
   ARRAY        [ 10 ][ 20 ][ 30 ][ 40 ]      contiguous; index is arithmetic

   LINKED LIST  ┌────┐   ┌────┐   ┌────┐   ┌────┐
        head ─> │ 10 │──>│ 20 │──>│ 30 │──>│ 40 │──> None
                └────┘   └────┘   └────┘   └────┘
                scattered anywhere in memory; you must WALK to reach one
```

That single difference produces every trade-off:

| | Array / `list` | Linked list |
|---|---|---|
| Access `data[i]` | **O(1)** | 🔴 **O(n)** — walk from the head |
| Insert/delete at the front | 🔴 O(n) | **O(1)** |
| Insert/delete at the back | O(1) amortised | O(1) *with a tail pointer* |
| Insert/delete **given the node** | O(n) | **O(1)** |
| Memory per element | one pointer | pointer + node object |
| Cache friendliness | ✅ excellent | 🔴 poor — jumps everywhere |
| Resizing | occasional copy | never |

**The trade in one line:** a linked list buys O(1) insertion and deletion by giving up O(1) access.

### 🔴 Be honest about when to use one

In production Python you will **almost never write a linked list**. `list` and `deque` are implemented in C and are faster for nearly everything, and a Python node object costs far more memory than a pointer in an array.

So why the whole notebook?

1. **Interviews ask about them constantly** — reversal, cycle detection and merging are standard questions, precisely because they test pointer reasoning rather than library knowledge.
2. **They teach reference manipulation**, which transfers directly to trees (**15.7**) and graphs (**15.9**) — both are linked structures with more pointers.
3. **They are inside things you use.** `collections.deque` is a doubly linked list of blocks; `functools.lru_cache` uses a doubly linked list; so do most LRU caches, undo stacks and adjacency lists.

> Learn them for the reasoning and the interview. Reach for `deque` at work.

In [ ]:
class Node:
    """One element: a value, and a reference to the next node."""

    __slots__ = ("value", "next")        # smaller objects - see 5.3

    def __init__(self, value, next_node=None):
        self.value = value
        self.next = next_node

    def __repr__(self):
        return f"Node({self.value!r})"


def build(values):
    """Create a list from an iterable, returning the head. O(n)."""
    head = None
    for value in reversed(list(values)):   # build backwards: each new node
        head = Node(value, head)           # points at what we had so far
    return head


def to_list(head, limit=1_000):
    """Collect values into a Python list. `limit` guards against cycles."""
    out = []
    node = head
    while node is not None and len(out) < limit:
        out.append(node.value)
        node = node.next
    return out


def show(head):
    values = to_list(head)
    return " -> ".join(map(str, values)) + " -> None" if values else "(empty)"


head = build([10, 20, 30, 40])
print(show(head))
print("head      :", head)
print("head.next :", head.next)
print("third     :", head.next.next)
print("\nThere is no index. Reaching the third node meant two hops - O(n).")

## Traversal, insertion, deletion

Every linked-list operation is the same three-step dance:

```
   1. walk to the node BEFORE the position you care about
   2. rewire the pointers
   3. do it in an order that does not lose the rest of the list
```

🔴 **Step 3 is where the bugs are.** Overwrite `prev.next` before you have saved `prev.next.next`, and the tail is unreachable — garbage collected, gone.

```
   insert 25 after 20:

        [20] ────────> [30]
                 [25]              1. new.next = prev.next     (25 -> 30)
        [20] ──> [25] ──> [30]     2. prev.next = new          (20 -> 25)
                                      🔴 THIS ORDER. Reversed, 30 is lost.
```

In [ ]:
def insert_after(node, value):
    """Insert a new node directly after `node`. O(1)."""
    node.next = Node(value, node.next)     # order matters: read then write


def push_front(head, value):
    """Insert at the head. O(1) - the linked list's superpower."""
    return Node(value, head)


def delete_value(head, target):
    """Remove the first node with this value. Returns the (possibly new) head."""
    # The head is a special case: there is no `prev` for it.
    if head is not None and head.value == target:
        return head.next
    prev = head
    while prev is not None and prev.next is not None:
        if prev.next.value == target:
            prev.next = prev.next.next     # unlink; GC reclaims the node
            return head
        prev = prev.next
    return head


head = build([10, 20, 30, 40])
print("start        :", show(head))

insert_after(head.next, 25)
print("insert 25    :", show(head))

head = push_front(head, 5)
print("push_front 5 :", show(head))

head = delete_value(head, 30)
print("delete 30    :", show(head))

head = delete_value(head, 5)
print("delete head  :", show(head))

head = delete_value(head, 999)
print("delete absent:", show(head))
print("\n🔴 Note how the HEAD needed its own branch in delete_value.")
print("   That special case is what the dummy-head trick removes.")

### The dummy head trick

Half the bugs in linked-list code come from the head being special: it has no predecessor, so every insert and delete needs an `if this is the head` branch.

**The fix:** create a throwaway node in front of the real head. Now *every* real node has a predecessor, and the special case disappears.

```
   [dummy] ──> [10] ──> [20] ──> [30] ──> None
      ^
      never holds data; return dummy.next at the end
```

> This is the single most useful linked-list technique in interviews. It turns fiddly code into uniform code, and interviewers recognise it immediately.

In [ ]:
def delete_all(head, target):
    """Remove EVERY node with this value - using a dummy head.

    Compare with delete_value above: no special case for the head at all.
    """
    dummy = Node(None, head)
    prev = dummy
    while prev.next is not None:
        if prev.next.value == target:
            prev.next = prev.next.next     # unlink, do NOT advance
        else:
            prev = prev.next
    return dummy.next                      # the real head, whatever it is now


for values, target in (([1, 2, 6, 3, 6, 4, 5, 6], 6),
                      ([7, 7, 7], 7),
                      ([1, 2, 3], 9)):
    head = build(values)
    head = delete_all(head, target)
    print(f"  remove {target} from {values} -> {to_list(head)}")

print("\n🔴 The [7,7,7] case removes every node INCLUDING the head, and")
print("   returns an empty list. Without a dummy head that needs its own")
print("   branch; with one it just works.")
print()
print("🔴 Also note: after unlinking we do NOT advance `prev` - the next")
print("   node might also match. Advancing there is a classic bug that")
print("   silently misses consecutive duplicates.")

## Reversing a linked list

The most-asked linked-list question there is. Three pointers, one pass, O(1) space.

```
   prev    current   next
   None  <─ [10]      [20] -> [30] -> None

   each step:  1. remember current.next          (or you lose the rest)
               2. current.next = prev            (flip the arrow)
               3. prev = current                 (shuffle both forward)
               4. current = the remembered next
```

🔴 Step 1 is not optional. The moment you write `current.next = prev`, the original `current.next` is gone unless you saved it.

| | Time | Space |
|---|---|---|
| Iterative | O(n) | **O(1)** |
| Recursive | O(n) | O(n) — the call stack (**15.1**) |

In [ ]:
def reverse_iterative(head):
    """Three pointers, one pass. O(n) time, O(1) space."""
    prev = None
    current = head
    while current is not None:
        following = current.next       # 1. save it FIRST
        current.next = prev            # 2. flip
        prev = current                 # 3. advance prev
        current = following            # 4. advance current
    return prev                        # prev is the new head


def reverse_recursive(node):
    """O(n) time, O(n) stack space - depth equals length (15.1)."""
    if node is None or node.next is None:
        return node                    # base case: the new head
    new_head = reverse_recursive(node.next)
    node.next.next = node              # make the next node point back at us
    node.next = None                   # and break our forward link
    return new_head


for values in ([1, 2, 3, 4, 5], [1], []):
    head = build(values)
    reversed_head = reverse_iterative(head)
    print(f"  iterative {str(values):<16} -> {to_list(reversed_head)}")

print()
for values in ([1, 2, 3, 4, 5], [1], []):
    head = build(values)
    reversed_head = reverse_recursive(head)
    print(f"  recursive {str(values):<16} -> {to_list(reversed_head)}")

print("\n🔴 The recursive version hits RecursionError on a long list:")
long_head = build(range(2_000))
try:
    reverse_recursive(long_head)
    print("   2,000 nodes: succeeded")
except RecursionError:
    print("   2,000 nodes: RecursionError - the stack ran out (15.1)")

long_head = build(range(2_000))
result = reverse_iterative(long_head)
print(f"   iterative on 2,000 nodes: fine, first value is {result.value}")

## 🔴 Floyd's cycle detection

A linked list can loop back on itself. Traversing one then never terminates — and this is not hypothetical: a bug in pointer rewiring produces exactly this.

**The tortoise and hare.** One pointer moves 1 step, the other 2.

- **No cycle:** the fast pointer reaches the end. Done.
- **Cycle:** the fast pointer laps the slow one and they **must** meet — the gap between them shrinks by exactly one each step, so it cannot be skipped over.

```
   1 -> 2 -> 3 -> 4 -> 5
             ^         |
             └─────────┘
```

| | Time | Space |
|---|---|---|
| Set of visited nodes | O(n) | O(n) |
| **Floyd's** | O(n) | **O(1)** |

**Finding where the cycle starts** is the follow-up. After they meet, move one pointer back to the head and advance both **one step at a time**; they meet at the entrance. That falls out of the algebra: the distance from the head to the entrance equals the distance from the meeting point to the entrance, modulo the cycle length.

In [ ]:
def has_cycle(head):
    """Floyd's tortoise and hare. O(n) time, O(1) space."""
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next               # 1 step
        fast = fast.next.next          # 2 steps
        if slow is fast:               # `is`, not `==` - identity, not value
            return True
    return False


def find_cycle_start(head):
    """Return the node where the cycle begins, or None."""
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            finder = head
            while finder is not slow:  # both move ONE step now
                finder, slow = finder.next, slow.next
            return finder
    return None


# a clean list
clean = build([1, 2, 3, 4, 5])
print("no cycle      :", has_cycle(clean), "| start:", find_cycle_start(clean))

# make node 5 point back at node 3
looped = build([1, 2, 3, 4, 5])
third = looped.next.next
last = looped
while last.next is not None:
    last = last.next
last.next = third

print("with cycle    :", has_cycle(looped), "| start:", find_cycle_start(looped))

# a node pointing at itself - the smallest possible cycle
self_loop = Node(99)
self_loop.next = self_loop
print("self-loop     :", has_cycle(self_loop), "| start:", find_cycle_start(self_loop))

print("\n🔴 `slow is fast`, never `==`. Two distinct nodes can hold equal")
print("   values; only identity means 'the same node'.")
print("\n🔴 And note `to_list` in this notebook takes a `limit` - without a")
print("   guard, printing a cyclic list hangs forever.")

### The fast/slow pointer does more than cycles

The same trick answers two other standard questions in one pass:

| Question | Trick |
|---|---|
| **Find the middle** | when fast reaches the end, slow is at the middle |
| **Find the nth from the end** | start fast n steps ahead, then move both together |

Both avoid the obvious two-pass solution (count the length, then walk again). One pass, O(1) space — and in an interview, saying "one pass" out loud is worth doing.

In [ ]:
def find_middle(head):
    """For even lengths this returns the SECOND middle - state which you mean."""
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
    return slow


def nth_from_end(head, n):
    """One pass, using a gap of n between two pointers."""
    lead = head
    for _ in range(n):
        if lead is None:
            return None                # the list is shorter than n
        lead = lead.next
    trail = head
    while lead is not None:
        lead, trail = lead.next, trail.next
    return trail


for values in ([1, 2, 3, 4, 5], [1, 2, 3, 4]):
    head = build(values)
    middle = find_middle(head)
    print(f"  middle of {values} -> {middle.value}")

print()
head = build([1, 2, 3, 4, 5])
for n in (1, 2, 5, 6):
    node = nth_from_end(head, n)
    print(f"  {n} from the end -> {node.value if node else 'out of range'}")

print("\n🔴 For an even-length list there are two middles. [1,2,3,4] gives 3")
print("   here. Which one is wanted is a question worth asking out loud.")

## Merging two sorted lists

The linked-list version of the merge step in merge sort (**15.10**) — and a common question in its own right.

Because you are only rewiring pointers, **no new nodes are needed**: the merge is O(1) extra space, which the array version cannot manage. That is worth saying explicitly if you are asked.

In [ ]:
def merge_sorted(a, b):
    """Merge two sorted lists by rewiring. O(n+m) time, O(1) extra space."""
    dummy = Node(None)
    tail = dummy
    while a is not None and b is not None:
        if a.value <= b.value:          # <= keeps it STABLE (15.10)
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a if a is not None else b     # attach whatever is left
    return dummy.next


cases = [
    ([1, 3, 5], [2, 4, 6]),
    ([1, 2, 3], []),
    ([], []),
    ([1, 1, 2], [1, 3]),
]
for left, right in cases:
    merged = merge_sorted(build(left), build(right))
    print(f"  {str(left):<12} + {str(right):<10} -> {to_list(merged)}")

print("\nNo nodes were allocated except the dummy - only pointers moved.")
print("The array equivalent needs O(n+m) space for the output.")

## Doubly linked lists, and where they actually live

Each node also points **backwards**:

```
   None <── ┌────┐ <──> ┌────┐ <──> ┌────┐ ──> None
            │ 10 │      │ 20 │      │ 30 │
            └────┘      └────┘      └────┘
```

The gain: **given a node, you can delete it in O(1)** — you have its predecessor. In a singly linked list you would have to walk from the head to find it.

That single property is what makes an **LRU cache** possible:

| Operation | How |
|---|---|
| find a key | `dict` → the node, O(1) (**15.2**) |
| mark it most-recently-used | unlink it and move it to the front, **O(1)** |
| evict the least-recently-used | remove the tail, **O(1)** |

A hash map alone cannot do the second and third. A linked list alone cannot do the first. Together they give O(1) for all three — which is exactly how `functools.lru_cache` and every production cache is built.

In [ ]:
class DNode:
    __slots__ = ("key", "value", "prev", "next")

    def __init__(self, key=None, value=None):
        self.key, self.value = key, value
        self.prev = self.next = None


class LRUCache:
    """dict for O(1) lookup + doubly linked list for O(1) reordering."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.entries = {}
        # Sentinel head and tail remove every edge case (the dummy trick again)
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head

    def _unlink(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def _push_front(self, node):
        node.prev, node.next = self.head, self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key, default=None):
        node = self.entries.get(key)
        if node is None:
            return default
        self._unlink(node)              # O(1) because it is doubly linked
        self._push_front(node)
        return node.value

    def put(self, key, value):
        node = self.entries.get(key)
        if node is not None:
            node.value = value
            self._unlink(node)
            self._push_front(node)
            return
        if len(self.entries) >= self.capacity:
            oldest = self.tail.prev     # O(1) eviction
            self._unlink(oldest)
            del self.entries[oldest.key]
        node = DNode(key, value)
        self.entries[key] = node
        self._push_front(node)

    def keys_mru_first(self):
        out, node = [], self.head.next
        while node is not self.tail:
            out.append(node.key)
            node = node.next
        return out


cache = LRUCache(capacity=3)
for key in ("a", "b", "c"):
    cache.put(key, key.upper())
print("after a,b,c      :", cache.keys_mru_first())

cache.get("a")                          # 'a' becomes most recent
print("after get('a')   :", cache.keys_mru_first())

cache.put("d", "D")                     # over capacity -> evict the LRU
print("after put('d')   :", cache.keys_mru_first())
print("'b' was evicted  :", cache.get("b") is None)
print("'a' survived     :", cache.get("a"))

print("\nEvery operation here is O(1): the dict finds the node, the doubly")
print("linked list moves it. Neither could do this alone.")

## Interview questions

**1. Reverse a linked list.** *(implemented above)*
> Three pointers, one pass, O(1) space. Offer the recursive version as an alternative and immediately note it is O(n) stack space.

**2. Detect a cycle, and find where it starts.** *(implemented above)*
> Floyd's. O(1) space beats the O(n) visited-set. Be ready to explain *why* they must meet: the gap closes by one per step.

**3. Find the middle node in one pass.** *(implemented above)*
> Fast/slow. Ask which middle they want for an even-length list.

**4. Remove the nth node from the end in one pass.** *(implemented above)*
> Two pointers with a gap of n. **Use a dummy head** — removing the first node is otherwise a special case.

**5. Merge two sorted linked lists.** *(implemented above)*
> Dummy head plus a tail pointer. O(1) extra space, unlike the array version.

**6. Merge k sorted lists.**
> A heap of the k current heads: O(N log k) (**15.8**). Or merge pairwise, halving k each round — also O(N log k). Merging one at a time is O(N·k) and is the answer to avoid.

**7. Detect whether two lists intersect.**
> Walk both; when one ends, restart it at the other's head. After at most two passes they align, because each pointer travels `lenA + lenB`. O(1) space.

**8. Remove duplicates from a sorted list.**
> One pass comparing `node.value` with `node.next.value`. If unsorted, a set — O(n) space.

**9. Check whether a list is a palindrome in O(1) space.**
> Find the middle, reverse the second half, compare, then restore. Everything in this notebook, combined.

**10. Why would you use a linked list over a Python list?**
> In real Python, almost never. Say so — then give the honest exceptions: O(1) insertion given a node, which is what makes an LRU cache work, and O(1) at both ends, which is what `deque` provides.

In [ ]:
# Question 9, since it combines three techniques from this notebook.
def is_palindrome_list(head):
    """O(n) time, O(1) space: find middle, reverse the back half, compare."""
    if head is None or head.next is None:
        return True

    # 1. find the middle (fast/slow)
    slow = fast = head
    while fast.next is not None and fast.next.next is not None:
        slow, fast = slow.next, fast.next.next

    # 2. reverse the second half
    second = reverse_iterative(slow.next)

    # 3. compare the two halves
    left, right = head, second
    result = True
    while right is not None:
        if left.value != right.value:
            result = False
            break
        left, right = left.next, right.next

    # 4. put it back - a courtesy the interviewer will notice
    slow.next = reverse_iterative(second)
    return result


for values in ([1, 2, 3, 2, 1], [1, 2, 2, 1], [1, 2, 3], [1], []):
    head = build(values)
    answer = is_palindrome_list(head)
    restored = to_list(head)
    print(f"  {str(values):<16} palindrome: {str(answer):<5} "
          f"list intact: {restored == values}")

print("\nStep 4 matters. Mutating the caller's data and not restoring it is")
print("a real defect - and remembering it out loud is a strong signal.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Overwriting `node.next` before saving it.** The rest of the list becomes unreachable. Save first, then rewire.
2. 🔴 **Comparing nodes with `==` instead of `is` in cycle detection.** Equal values do not mean the same node.
3. 🔴 **Advancing `prev` after unlinking.** The next node may also match - you will silently miss consecutive duplicates.
4. 🔴 **Traversing without a guard when a cycle is possible.** The loop never ends.
5. **Forgetting the head is special.** Use a dummy head and the special case vanishes.
6. **Recursive solutions on long lists.** `RecursionError` at ~1000 nodes (**15.1**).
7. **Not checking `fast.next` as well as `fast`.** `fast.next.next` raises `AttributeError` on an even-length list otherwise.
8. **Assuming `len()` is available.** A linked list has no stored length; counting is O(n).
9. **Mutating the input and not restoring it** when the question did not ask you to.

## Best Practices

- Use a dummy head node for anything that inserts or deletes.
- Save `next` before rewiring, every time.
- Use `is` for node identity, `==` only for values.
- Prefer iterative solutions; recursion costs O(n) stack.
- Test the empty list, one node, and two nodes - that is where the bugs are.
- Keep a tail pointer if you append often; otherwise appending is O(n).
- Name pointers for their role: `prev`, `current`, `slow`, `fast`, `tail`.
- In real Python, reach for `deque` or `list` - and be able to say why.

## Practice Exercises

Try these before moving on.

1. Add a `length` field maintained by insert and delete so `len()` is O(1). What does that cost, and when is it worth it?
2. Implement `remove_nth_from_end` using a dummy head, and test removing the *first* node - the case the dummy exists for.
3. 🔴 Implement 'merge k sorted lists' twice: merging one at a time, and pairwise. Compare the operation counts for k = 16.
4. Implement 'add two numbers represented as linked lists' where digits are stored in reverse order. Watch the final carry.
5. Write `detect_intersection(a, b)` using the two-pointer restart trick, and explain in one sentence why both pointers travel the same total distance.
6. Implement a singly linked list class with `__iter__`, `__len__` and `__repr__` so it behaves like a Python container (**05 OOPs**).
7. 🔴 Benchmark your linked list against `collections.deque` for 100,000 `appendleft` operations. Explain the gap using **15.2**.
8. Implement an LRU cache with `OrderedDict` instead, using `move_to_end`. Which version would you ship, and which would you write in an interview?